# Demo of KG with LLM applications

## KG construction with Neo4j 

In [ ]:
import os, sys
import re
# Settings #
cwd = os.getcwd()
frameworkDir = os.path.abspath(os.path.join(cwd, os.pardir, os.pardir, 'src'))
sys.path.append(frameworkDir)
print(f"Add path: {frameworkDir} to system path")
#######################
# Internal Modules #
from dackar.knowledge_graph.py2neo import Py2Neo
from dackar.knowledge_graph.graph_utils import set_neo4j_import_folder

# uri = "bolt://localhost:7687" # for a single instance
uri = "neo4j://localhost:7687" # for a cluster
pwd = "123456789" # user need to provide the DBMS database password

py2neo = Py2Neo(uri=uri, user='neo4j', pwd=pwd)
py2neo.reset()
# py2neo.close()

In [ ]:
### Load MBSE model node data
file_path = 'test_nodes.csv'
label = 'MBSE'
attribute = {'nodeId':'nodeId', 'label':'label', 'ID':'ID', 'type':'type'}
py2neo.load_csv_for_nodes(file_path, label, attribute)
### Load MBSE model relationship data
file_path = 'test_edges.csv'
l1='MBSE'
p1={'nodeId':'sourceNodeId'}
l2='MBSE'
p2 ={'nodeId':'targetNodeId'}
lr = 'MBSE_link'
pr = {'prop':'type'}
py2neo.load_csv_for_relations(file_path, l1, p1, l2, p2, lr, pr)
### Load monitoring data
file_path = 'test_monit_vars.csv'
label = 'monitor_var'
attribute = {'nodeId':'varID'}
py2neo.load_csv_for_nodes(file_path, label, attribute)
### Load monitoring relationship data and link to MBSE model data
file_path = 'test_monit_vars.csv'
l1='monitor_var'
p1={'nodeId':'varID'}
l2='MBSE'
p2 ={'nodeId':'MBSE_ID'}
lr = 'monitoring'
pr = None
py2neo.load_csv_for_relations(file_path, l1, p1, l2, p2, lr, pr)
### Load anomaly detection data
file_path = 'test_AD.csv'
label = 'anomaly_detect'
attribute = {'nodeId':'AD_ID', 'type':'type'}
py2neo.load_csv_for_nodes(file_path, label, attribute)
### Load anomaly detection relation data and link to monitoring data
file_path = 'test_AD.csv'
l1='anomaly_detect'
p1={'nodeId':'AD_ID'}
l2='monitor_var'
p2 ={'nodeId':'var_ID'}
lr = 'input_from'
pr = None
py2neo.load_csv_for_relations(file_path, l1, p1, l2, p2, lr, pr)
### Load anomalies
file_path = 'test_anomalies.csv'
label = 'anomaly'
attribute = {'nodeId':'anom_ID', 'time_initial':'t_in', 'time_final':'t_fin'}
py2neo.load_csv_for_nodes(file_path, label, attribute)
### Load anomalies relation data and link to anomaly detection method
file_path = 'test_anomalies.csv'
l1='anomaly'
p1={'nodeId':'anom_ID'}
l2='anomaly_detect'
p2 ={'nodeId':'AD_ID'}
lr = 'detected_by'
pr = None
py2neo.load_csv_for_relations(file_path, l1, p1, l2, p2, lr, pr)

## LLM Integration

In [ ]:
# Install Dackar libraries first
# Then install LLM related libraries
# !pip install openai langchain langchain_openai langchain_community langchain_neo4j
# !pip install langchain-core ollama langchain-ollama

In [ ]:
from langchain_ollama.chat_models import ChatOllama

llm = ChatOllama(
    model="gpt-oss:20b",          # must be pulled in Ollama
    temperature=0,             # pass any Ollama params here
    base_url="http://localhost:11434",  # default; override if needed
)

print(llm.invoke("Explain RAG in one sentence.").content)